# Flickr30k T3→S3 INT8 Candidate Preflight

Sàng lọc hai ứng viên: `(1)` encoder và decoder dynamic INT8; `(2)` encoder
FP32 kết hợp decoder dynamic INT8. Cả hai được kiểm tra end-to-end trên cùng
năm đầu vào tổng hợp. Ứng viên nhỏ nhất vượt toàn bộ gate mới được giữ lại.
Notebook không đọc Train, Validation, ảnh Flickr30k hoặc Held-out Test.

## Kaggle input bắt buộc — đúng 2 input

1. Output `F30K_T3S3_ONNX_FP32`.
2. Dataset `ONNX_Runtime_Python313_Offline`.

Runtime: CPU/Accelerator None, Internet Off, dự kiến 3–10 phút.

Gate preflight không thay đổi: model reload được, giảm ít nhất 10% dung lượng,
cosine encoder và decoder tối thiểu 0,995, top-1 agreement tối thiểu 80%.
Full dynamic INT8 không được hạ ngưỡng nếu thất bại; mixed precision chỉ được
chọn nếu tự nó vượt đầy đủ gate.

## Cell 1 — Authorization and FP32 integrity

In [1]:
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

EXPERIMENT_ID = "F30K_T3S3_INT8_Preflight"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

FP32_RESULT_PATH = unique_named("F30K_T3S3_ONNX_FP32_RESULT.json")
FP32_RESULT = read_json(FP32_RESULT_PATH)
assert FP32_RESULT["status"] == "passed"
assert FP32_RESULT["format"] == "ONNX FP32"
assert FP32_RESULT["source"]["seed"] == 42
assert FP32_RESULT["validation_data_read"] is False
assert FP32_RESULT["test_set_status"] == "sealed"
assert FP32_RESULT["test_set_used"] is False

ENCODER_FP32_INPUT = (
    FP32_RESULT_PATH.parent / "f30k_t3s3_seed42_encoder_fp32.onnx"
)
DECODER_FP32_INPUT = (
    FP32_RESULT_PATH.parent / "f30k_t3s3_seed42_decoder_fp32.onnx"
)
for path in [ENCODER_FP32_INPUT, DECODER_FP32_INPUT]:
    artifact = FP32_RESULT["artifacts"][path.name]
    assert path.stat().st_size == artifact["bytes"]
    assert sha256(path) == artifact["sha256"]

# onnxruntime.quantization thực hiện shape inference và tạo file
# "*-inferred.onnx" cạnh model nguồn. /kaggle/input là read-only, vì vậy phải
# staging model sang /kaggle/working trước khi gọi quantize_dynamic.
ENCODER_FP32 = WORK_ROOT / ENCODER_FP32_INPUT.name
DECODER_FP32 = WORK_ROOT / DECODER_FP32_INPUT.name
shutil.copy2(ENCODER_FP32_INPUT, ENCODER_FP32)
shutil.copy2(DECODER_FP32_INPUT, DECODER_FP32)
assert sha256(ENCODER_FP32) == sha256(ENCODER_FP32_INPUT)
assert sha256(DECODER_FP32) == sha256(DECODER_FP32_INPUT)

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn dataset ONNX_Runtime_Python313_Offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
assert importlib.util.find_spec("onnx") is not None
assert importlib.util.find_spec("onnxruntime") is not None

import onnx
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

print("Preflight authorization : PASSED")
print("Source format           : ONNX FP32 / opset",
      FP32_RESULT["opset"])
print("Source total MiB        :",
      (ENCODER_FP32.stat().st_size + DECODER_FP32.stat().st_size) / 1024**2)
print("ONNX Runtime            :", ort.__version__)
print("Validation / Test read  : False / False")

Preflight authorization : PASSED
Source format           : ONNX FP32 / opset 17
Source total MiB        : 46.97272872924805
ONNX Runtime            : 1.28.0
Validation / Test read  : False / False


## Cell 2 — Dynamic per-channel QInt8 quantization

In [2]:
ENCODER_INT8 = WORK_ROOT / "f30k_t3s3_seed42_encoder_dynamic_int8.onnx"
DECODER_INT8 = WORK_ROOT / "f30k_t3s3_seed42_decoder_dynamic_int8.onnx"

quantization_arguments = {
    "per_channel": True,
    "reduce_range": False,
    "weight_type": QuantType.QInt8,
}
print("[INT8] Quantizing encoder...", flush=True)
quantize_dynamic(
    model_input=str(ENCODER_FP32),
    model_output=str(ENCODER_INT8),
    **quantization_arguments,
)
print("[INT8] Quantizing decoder...", flush=True)
quantize_dynamic(
    model_input=str(DECODER_FP32),
    model_output=str(DECODER_INT8),
    **quantization_arguments,
)
onnx.checker.check_model(onnx.load(str(ENCODER_INT8)))
onnx.checker.check_model(onnx.load(str(DECODER_INT8)))

fp32_bytes = ENCODER_FP32.stat().st_size + DECODER_FP32.stat().st_size
int8_bytes = ENCODER_INT8.stat().st_size + DECODER_INT8.stat().st_size
reduction_fraction = 1.0 - int8_bytes / fp32_bytes
assert int8_bytes < fp32_bytes
assert reduction_fraction >= 0.10
print("INT8 checker            : PASSED")
print("FP32 / INT8 MiB         :", fp32_bytes / 1024**2,
      "/", int8_bytes / 1024**2)
print("Size reduction          :", reduction_fraction)

[INT8] Quantizing encoder...


[INT8] Quantizing decoder...


INT8 checker            : PASSED
FP32 / INT8 MiB         : 46.97272872924805 / 12.790532112121582
Size reduction          : 0.7277030213457149


## Cell 3 — Full-INT8 versus mixed-precision screening

In [3]:
providers = ["CPUExecutionProvider"]
sessions = {
    "encoder_fp32": ort.InferenceSession(str(ENCODER_FP32), providers=providers),
    "decoder_fp32": ort.InferenceSession(str(DECODER_FP32), providers=providers),
    "encoder_int8": ort.InferenceSession(str(ENCODER_INT8), providers=providers),
    "decoder_int8": ort.InferenceSession(str(DECODER_INT8), providers=providers),
}

def cosine(left, right):
    left = np.asarray(left, dtype=np.float64).reshape(-1)
    right = np.asarray(right, dtype=np.float64).reshape(-1)
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    assert denominator > 0
    return float(np.dot(left, right) / denominator)

full_int8_rows = []
mixed_rows = []
for case_index, valid_length in enumerate([1, 4, 8, 16, 30]):
    rng = np.random.default_rng(20261026 + case_index)
    image = rng.standard_normal((1, 3, 224, 224), dtype=np.float32)
    tokens = np.zeros((1, 31), dtype=np.int64)
    tokens[0, 0] = 1
    if valid_length > 1:
        tokens[0, 1:valid_length] = rng.integers(
            4, 7456, size=(valid_length - 1,), dtype=np.int64
        )
    mask = np.zeros((1, 31), dtype=np.bool_)
    mask[0, :valid_length] = True

    memory_fp32 = sessions["encoder_fp32"].run(
        ["memory"], {"images": image}
    )[0]
    memory_int8 = sessions["encoder_int8"].run(
        ["memory"], {"images": image}
    )[0]
    logits_fp32 = sessions["decoder_fp32"].run(
        ["logits"],
        {
            "token_ids": tokens,
            "memory": memory_fp32,
            "attention_mask": mask,
        },
    )[0]
    logits_full_int8 = sessions["decoder_int8"].run(
        ["logits"],
        {
            "token_ids": tokens,
            "memory": memory_int8,
            "attention_mask": mask,
        },
    )[0]
    logits_mixed = sessions["decoder_int8"].run(
        ["logits"],
        {
            "token_ids": tokens,
            "memory": memory_fp32,
            "attention_mask": mask,
        },
    )[0]
    next_fp32 = logits_fp32[:, valid_length - 1]
    next_full_int8 = logits_full_int8[:, valid_length - 1]
    next_mixed = logits_mixed[:, valid_length - 1]
    encoder_cosine = cosine(memory_fp32, memory_int8)
    encoder_mae = float(np.mean(np.abs(memory_fp32 - memory_int8)))
    full_int8_rows.append({
        "case": case_index,
        "valid_token_length": valid_length,
        "encoder_cosine": encoder_cosine,
        "encoder_mean_abs_error": encoder_mae,
        "decoder_next_logits_cosine": cosine(next_fp32, next_full_int8),
        "decoder_next_logits_mean_abs_error": float(
            np.mean(np.abs(next_fp32 - next_full_int8))
        ),
        "next_token_top1_equal": bool(np.array_equal(
            next_fp32.argmax(axis=-1), next_full_int8.argmax(axis=-1)
        )),
    })
    mixed_rows.append({
        "case": case_index,
        "valid_token_length": valid_length,
        "encoder_cosine": 1.0,
        "encoder_mean_abs_error": 0.0,
        "decoder_next_logits_cosine": cosine(next_fp32, next_mixed),
        "decoder_next_logits_mean_abs_error": float(
            np.mean(np.abs(next_fp32 - next_mixed))
        ),
        "next_token_top1_equal": bool(np.array_equal(
            next_fp32.argmax(axis=-1), next_mixed.argmax(axis=-1)
        )),
    })
    assert np.isfinite(memory_int8).all()
    assert np.isfinite(next_full_int8).all()
    assert np.isfinite(next_mixed).all()

full_int8_bytes = ENCODER_INT8.stat().st_size + DECODER_INT8.stat().st_size
mixed_bytes = ENCODER_FP32.stat().st_size + DECODER_INT8.stat().st_size

def evaluate_candidate(name, rows, total_bytes):
    minimum_encoder_cosine = min(row["encoder_cosine"] for row in rows)
    minimum_decoder_cosine = min(
        row["decoder_next_logits_cosine"] for row in rows
    )
    top1_agreement = (
        sum(row["next_token_top1_equal"] for row in rows) / len(rows)
    )
    size_reduction = 1.0 - total_bytes / fp32_bytes
    gates = {
        "minimum_encoder_cosine_required": 0.995,
        "minimum_encoder_cosine_observed": minimum_encoder_cosine,
        "minimum_decoder_cosine_required": 0.995,
        "minimum_decoder_cosine_observed": minimum_decoder_cosine,
        "minimum_top1_agreement_required": 0.80,
        "top1_agreement_observed": top1_agreement,
        "minimum_size_reduction_required": 0.10,
        "size_reduction_observed": size_reduction,
    }
    passed = (
        minimum_encoder_cosine >= gates["minimum_encoder_cosine_required"]
        and minimum_decoder_cosine >= gates["minimum_decoder_cosine_required"]
        and top1_agreement >= gates["minimum_top1_agreement_required"]
        and size_reduction >= gates["minimum_size_reduction_required"]
    )
    return {
        "name": name,
        "status": "passed" if passed else "rejected",
        "total_bytes": total_bytes,
        "total_MiB": total_bytes / 1024**2,
        "gates": gates,
        "synthetic_parity_cases": rows,
    }

candidates = [
    evaluate_candidate(
        "full_dynamic_int8", full_int8_rows, full_int8_bytes
    ),
    evaluate_candidate(
        "mixed_fp32_encoder_dynamic_int8_decoder",
        mixed_rows,
        mixed_bytes,
    ),
]
passing_candidates = [
    candidate for candidate in candidates if candidate["status"] == "passed"
]
assert passing_candidates, candidates
selected = min(passing_candidates, key=lambda candidate: candidate["total_bytes"])
assert selected["name"] == "mixed_fp32_encoder_dynamic_int8_decoder", selected

reload_sessions = [
    ort.InferenceSession(str(path), providers=providers)
    for path in [ENCODER_FP32, DECODER_INT8]
]
assert len(reload_sessions) == 2
print("Full dynamic INT8 gate :", candidates[0]["status"].upper())
print("Mixed precision gate   :", candidates[1]["status"].upper())
print("Selected candidate     :", selected["name"])
print("Selected total MiB     :", selected["total_MiB"])
print("Selected reduction     :",
      selected["gates"]["size_reduction_observed"])
print("Selected decoder cosine:",
      selected["gates"]["minimum_decoder_cosine_observed"])
print("Selected top-1         :",
      selected["gates"]["top1_agreement_observed"])

Full dynamic INT8 gate : REJECTED
Mixed precision gate   : PASSED
Selected candidate     : mixed_fp32_encoder_dynamic_int8_decoder
Selected total MiB     : 27.182700157165527
Selected reduction     : 0.42130889789589887
Selected decoder cosine: 0.9997634484346791
Selected top-1         : 1.0


## Cell 4 — Contract, artifacts and next-phase gate

In [4]:
selected_artifacts = {
    ENCODER_FP32.name: {
        "precision": "FP32",
        "bytes": ENCODER_FP32.stat().st_size,
        "sha256": sha256(ENCODER_FP32),
    },
    DECODER_INT8.name: {
        "precision": "dynamic QInt8 weights",
        "bytes": DECODER_INT8.stat().st_size,
        "sha256": sha256(DECODER_INT8),
    },
}
result = {
    "schema_version": "f30k_t3s3_mixed_precision_preflight_v2",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "source": {
        "experiment_id": FP32_RESULT["experiment_id"],
        "result_sha256": sha256(FP32_RESULT_PATH),
        "encoder_sha256": sha256(ENCODER_FP32_INPUT),
        "decoder_sha256": sha256(DECODER_FP32_INPUT),
        "seed": 42,
        "method": "Direct Logit-KD",
    },
    "screened_candidates": candidates,
    "selection_policy": (
        "Smallest candidate passing the predeclared cosine, top-1, reload, "
        "finite-output, and size-reduction gates."
    ),
    "selected_candidate": selected["name"],
    "selected_precision": {
        "encoder": "FP32",
        "decoder": "dynamic per-channel QInt8 weights",
    },
    "selected_size": {
        "fp32_reference_bytes": fp32_bytes,
        "selected_bytes": selected["total_bytes"],
        "fp32_reference_MiB": fp32_bytes / 1024**2,
        "selected_MiB": selected["total_MiB"],
        "reduction_fraction": selected["gates"]["size_reduction_observed"],
    },
    "artifacts": selected_artifacts,
    "validation_data_read": False,
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_authorized": "F30K_T3S3_INT8_Validation",
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
contract_path = WORK_ROOT / "F30K_T3S3_INT8_CONTRACT.json"
write_json(result_path, result)
write_json(contract_path, result)

# Giữ encoder FP32 và decoder INT8 của ứng viên thắng. Loại encoder INT8 đã
# thất bại gate cùng decoder FP32 staging.
del sessions
del reload_sessions
ENCODER_INT8.unlink()
DECODER_FP32.unlink()

artifacts = []
for path in [ENCODER_FP32, DECODER_INT8, result_path, contract_path]:
    artifacts.append({
        "relative_path": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_mixed_preflight_manifest_v2",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": artifacts,
    "validation_data_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Status                  : COMPLETE")
print("Selected precision      : FP32 encoder + dynamic INT8 decoder")
print("Selected total MiB      :", result["selected_size"]["selected_MiB"])
print("Reduction               :",
      result["selected_size"]["reduction_fraction"])
print("Validation / Test read  : False / False")
print("Next                    :", result["next_phase_authorized"])

Status                  : COMPLETE
Selected precision      : FP32 encoder + dynamic INT8 decoder
Selected total MiB      : 27.182700157165527
Reduction               : 0.42130889789589887
Validation / Test read  : False / False
Next                    : F30K_T3S3_INT8_Validation
